# Build `data_sat_soil_20d`

This notebook builds the agreed flat dataset documented in `data_sat_soil_20d_dataset.md`.

It intentionally does **not** train a model. The output is a model-ready table where one row is:

```text
county-year x 10d window x wheat/soil cell
```

For each included county-year, every wheat/soil cell appears once in every one of the 14 ten-day windows. Missing satellite cell-windows are kept as real rows with `observed_satellite = 0`, `sat_count_pixels = 0`, and all `sat_*` features set to zero.

## Requirements

Run this notebook in an environment with at least:

```text
pandas
numpy
geopandas
pyarrow or fastparquet   # only needed for the final Parquet cell
```

The notebook assumes it is located in the project root `Code_minimum`.

In [1]:
from __future__ import annotations

import json
import os
import re
import unicodedata
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from pyproj import Transformer
import warnings
from pandas.errors import PerformanceWarning

warnings.filterwarnings('ignore', category=PerformanceWarning)

## Configuration

Keep paths and policy constants in one place. The window policy mirrors the existing 10d preprocessing: `7.4` through `31.8`, stepping by 10 days, which gives 14 windows.

In [2]:
BASE_DIR = Path.cwd()

YEARS = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2025]

FUSED_DIR_TEMPLATE = "Data_Fused_{year}"
WINDOWED_DIR_TEMPLATE = "Data_Windowed_{year}_20d"
SOIL_DIR = BASE_DIR / "Data_Astatic_wheat_soil_cells"
SHAPE_PATH = BASE_DIR / "Data_Ashape" / "gadm36_USA_2.shp"
YIELD_CSV = BASE_DIR / "Ertragsdaten_USA.csv"

OUTPUT_CSV = BASE_DIR / "data_sat_soil_20d.csv"
OUTPUT_PARQUET = BASE_DIR / "data_sat_soil_20d.parquet"
OUTPUT_SCHEMA_JSON = BASE_DIR / "data_sat_soil_20d_schema.json"
OUTPUT_MANIFEST_CSV = BASE_DIR / "data_sat_soil_20d_manifest.csv"
OUTPUT_CHUNK_DIR = BASE_DIR / "data_sat_soil_20d_chunks"
OUTPUT_UNMATCHED_CENTROIDS_CSV = BASE_DIR / "data_sat_soil_20d_unmatched_centroids.csv"
PATCHED_SOIL_CSV = BASE_DIR / "soil_points_filled.csv"

WINDOW_START_DM = "7.4"
WINDOW_END_DM = "24.8"
WINDOW_FREQUENCY_DAYS = 20
EXPECTED_N_WINDOWS = 7

# Coordinate join precision used because fused satellite and soil-cell lon/lat align after rounding.
JOIN_COORD_DECIMALS = 4

# The single satellite count column kept in the final table.
PREFERRED_SAT_COUNT_SOURCE = "B2_count"

# Keep this strict at first. If it raises, inspect the schema differences before relaxing it.
STRICT_REQUIRE_COMMON_SAT_COLUMNS = True
STRICT_REQUIRE_COMMON_SOIL_COLUMNS = True

# Keep edge cells, including static_wheat_pixels_incounty == 0.
# Missing soil is patched from soil_points_filled.csv when possible, then tracked in the manifest.
POSITIVE_INCOUNTY_WHEAT_THRESHOLD = 1
USE_PATCHED_SOIL = True
INCLUDE_PATCHED_SAT_ONLY_CELLS = True

# Threaded chunk build. The final CSV is still concatenated in deterministic county-year order.
MAX_WORKERS = min(8, os.cpu_count() or 1)

# Write progress every N county-years during the build.
PROGRESS_EVERY = 50

# Used only for rare satellite-only cells that are absent from the static soil-cell table.
LONLAT_TO_5070 = Transformer.from_crs("EPSG:4326", "EPSG:5070", always_xy=True)

## Small Helpers

These helpers are deliberately plain. They parse the date format, define the 10-day windows, normalize names for centroid joins, and define stable path conventions.

In [3]:
@dataclass(frozen=True)
class CountyYear:
    year: int
    county_id: str
    state_id: int
    county_num: int
    fused_path: Path
    soil_path: Path
    windowed_path: Path


def parse_day_month(value: object, dummy_year: int = 2000) -> date:
    """Parse the project date format, e.g. '7.4' = 7 April."""
    text = str(value).strip()
    if not text or text.lower() == "nan":
        raise ValueError(f"Cannot parse day.month value: {value!r}")
    day_text, month_text = text.split(".", 1)
    return date(dummy_year, int(month_text), int(day_text))


def make_windows(start_dm: str, end_dm: str, frequency_days: int) -> pd.DataFrame:
    """Return a DataFrame with zero-based window_idx and fixed 10-day boundaries."""
    start = parse_day_month(start_dm)
    end = parse_day_month(end_dm)
    rows = []
    current = start
    idx = 0
    while current + timedelta(days=frequency_days - 1) <= end:
        window_end = current + timedelta(days=frequency_days - 1)
        rows.append(
            {
                "window_idx": idx,
                "date_start": f"{current.day}.{current.month}",
                "date_end": f"{window_end.day}.{window_end.month}",
                "_date_start_obj": current,
                "_date_end_obj": window_end,
            }
        )
        current = window_end + timedelta(days=1)
        idx += 1
    return pd.DataFrame(rows)


WINDOWS = make_windows(WINDOW_START_DM, WINDOW_END_DM, WINDOW_FREQUENCY_DAYS)
assert len(WINDOWS) == EXPECTED_N_WINDOWS, WINDOWS
WINDOWS[["window_idx", "date_start", "date_end"]]

,window_idx,date_start,date_end
0,0,7.4,16.4
1,1,17.4,26.4
2,2,27.4,6.5
3,3,7.5,16.5
4,4,17.5,26.5
5,5,27.5,5.6
6,6,6.6,15.6
7,7,16.6,25.6
8,8,26.6,5.7
9,9,6.7,15.7


In [4]:
def normalize_name(value: object) -> str:
    """Normalize names for joining NASS yield rows to GADM county shapes."""
    text = "" if pd.isna(value) else str(value)
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = text.lower()
    text = text.replace("&", " and ")
    text = re.sub(r"\b(county|parish|borough|municipality|census area|city and borough|city)\b", " ", text)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def add_join_coordinates(df: pd.DataFrame, lon_col: str = "longitude", lat_col: str = "latitude") -> pd.DataFrame:
    """Add rounded coordinate keys used to align soil cells and fused satellite cells."""
    out = df.copy()
    out["longitude_debug"] = out[lon_col].round(JOIN_COORD_DECIMALS)
    out["latitude_debug"] = out[lat_col].round(JOIN_COORD_DECIMALS)
    return out


def county_id_from_parts(state_id: int, county_num: int) -> str:
    return f"{int(state_id)}_{int(county_num)}"


def fused_path_for(year: int, state_id: int, county_num: int) -> Path:
    return BASE_DIR / FUSED_DIR_TEMPLATE.format(year=year) / f"Harvester_{year}_{state_id}_{county_num}.csv"


def soil_path_for(year: int, county_id: str) -> Path:
    return SOIL_DIR / f"county_{county_id}_{year}_static_wheat_soil_cells.csv"

## Discover Included County-Years

The whitelist comes from `Data_Windowed_{year}_20d`. If a county-year is not there, it is intentionally not included.

In [5]:
def discover_county_years() -> list[CountyYear]:
    records: list[CountyYear] = []

    for year in YEARS:
        windowed_dir = BASE_DIR / WINDOWED_DIR_TEMPLATE.format(year=year)
        if not windowed_dir.exists():
            print(f"Missing windowed folder, skipping year {year}: {windowed_dir}")
            continue

        for windowed_path in sorted(windowed_dir.glob(f"Harvester_{year}_*_*.csv")):
            parts = windowed_path.stem.split("_")
            if len(parts) != 4:
                print(f"Skipping unexpected filename: {windowed_path.name}")
                continue

            state_id = int(parts[2])
            county_num = int(parts[3])
            county_id = county_id_from_parts(state_id, county_num)
            fused_path = fused_path_for(year, state_id, county_num)
            soil_path = soil_path_for(year, county_id)

            records.append(
                CountyYear(
                    year=year,
                    county_id=county_id,
                    state_id=state_id,
                    county_num=county_num,
                    fused_path=fused_path,
                    soil_path=soil_path,
                    windowed_path=windowed_path,
                )
            )

    return sorted(records, key=lambda r: (r.year, r.state_id, r.county_num))


COUNTY_YEARS = discover_county_years()
print(f"Discovered {len(COUNTY_YEARS):,} whitelisted county-years")
pd.DataFrame([r.__dict__ for r in COUNTY_YEARS]).head()

Discovered 1,872 whitelisted county-years


,year,county_id,state_id,county_num,fused_path,soil_path,windowed_path
0,2016,5_39,5,39,\Documents\Semester_9\Master\Cod...,\Documents\Semester_9\Master\Cod...,\Documents\Semester_9\Master\Cod...
1,2016,6_1,6,1,\Documents\Semester_9\Master\Cod...,\Documents\Semester_9\Master\Cod...,\Documents\Semester_9\Master\Cod...
2,2016,6_22,6,22,\Documents\Semester_9\Master\Cod...,\Documents\Semester_9\Master\Cod...,\Documents\Semester_9\Master\Cod...
3,2016,6_32,6,32,\Documents\Semester_9\Master\Cod...,\Documents\Semester_9\Master\Cod...,\Documents\Semester_9\Master\Cod...
4,2016,6_33,6,33,\Documents\Semester_9\Master\Cod...,\Documents\Semester_9\Master\Cod...,\Documents\Semester_9\Master\Cod...


## Load Yield Targets

Yield is joined by `(year, county_id)` and repeated for every row of that county-year. Historical yield is intentionally not included.

In [6]:
def load_yield_lookup() -> tuple[dict[tuple[int, str], float], pd.DataFrame]:
    y = pd.read_csv(YIELD_CSV)
    y["lev1"] = pd.to_numeric(y["lev1"], errors="coerce")
    y["lev2"] = pd.to_numeric(y["lev2"], errors="coerce")

    # County rows have real state and county numeric ids. National/state summary rows have -1.
    county_rows = y[(y["lev1"] > 0) & (y["lev2"] > 0)].copy()
    county_rows["county_id"] = [county_id_from_parts(a, b) for a, b in zip(county_rows["lev1"], county_rows["lev2"])]

    lookup: dict[tuple[int, str], float] = {}
    for year in YEARS:
        col = f"Yield {year}"
        if col not in county_rows.columns:
            continue
        for county_id, value in county_rows[["county_id", col]].dropna().itertuples(index=False):
            lookup[(year, county_id)] = float(value)

    return lookup, county_rows


YIELD_LOOKUP, YIELD_COUNTIES = load_yield_lookup()
print(f"Loaded {len(YIELD_LOOKUP):,} county-year yield targets")
YIELD_COUNTIES[["county_id", "Name", "lev1", "lev2"]].head()

Loaded 8,284 county-year yield targets


,county_id,Name,lev1,lev2
2,1_1,Autauga,1,1
3,1_2,Baldwin,1,2
4,1_3,Barbour,1,3
5,1_4,Bibb,1,4
6,1_5,Blount,1,5


## County Polygon Centroids

Use the true GADM county polygon centroid from `Data_Ashape/gadm36_USA_2.shp`.

The centroid is computed after projecting shapes to EPSG:5070, so the output coordinates are meters. Cell offsets are then:

```text
distance_center_x_km = (cell_x_5070 - county_centroid_x_5070) / 1000
distance_center_y_km = (cell_y_5070 - county_centroid_y_5070) / 1000
```

The join from yield counties to GADM uses normalized state/county names. The cell below writes unmatched names to a CSV and raises if any whitelisted county is missing a centroid.

In [7]:
# Optional manual fixes can be added here if GADM and NASS spell a county differently.
# Keys and values are normalized names, scoped by normalized state name.
COUNTY_NAME_OVERRIDES = {
    # Example:
    # ("louisiana", "la salle"): "lasalle",
}


def load_county_centroids() -> pd.DataFrame:
    y = pd.read_csv(YIELD_CSV)
    y["lev1"] = pd.to_numeric(y["lev1"], errors="coerce")
    y["lev2"] = pd.to_numeric(y["lev2"], errors="coerce")

    state_rows = y[(y["lev1"] > 0) & (y["lev2"] == -1)].copy()
    state_name_by_id = dict(zip(state_rows["lev1"].astype(int), state_rows["Name"].astype(str)))

    county_rows = y[(y["lev1"] > 0) & (y["lev2"] > 0)].copy()
    county_rows["state_id"] = county_rows["lev1"].astype(int)
    county_rows["county_num"] = county_rows["lev2"].astype(int)
    county_rows["county_id"] = [county_id_from_parts(a, b) for a, b in zip(county_rows["state_id"], county_rows["county_num"])]
    county_rows["state_name"] = county_rows["state_id"].map(state_name_by_id)
    county_rows["state_norm"] = county_rows["state_name"].map(normalize_name)
    county_rows["county_norm"] = county_rows["Name"].map(normalize_name)
    county_rows["county_norm"] = [
        COUNTY_NAME_OVERRIDES.get((state, county), county)
        for state, county in zip(county_rows["state_norm"], county_rows["county_norm"])
    ]

    shapes = gpd.read_file(SHAPE_PATH)
    # GADM USA level-2 commonly uses NAME_1 for state and NAME_2 for county.
    state_col = "NAME_1"
    county_col = "NAME_2"
    if state_col not in shapes.columns or county_col not in shapes.columns:
        raise ValueError(f"Expected GADM columns {state_col!r} and {county_col!r}. Found: {list(shapes.columns)}")

    shapes_5070 = shapes.to_crs("EPSG:5070")
    centroids = shapes_5070.geometry.centroid

    shape_keys = pd.DataFrame(
        {
            "state_norm": shapes[state_col].map(normalize_name),
            "county_norm": shapes[county_col].map(normalize_name),
            "centroid_x_5070": centroids.x.astype("float64"),
            "centroid_y_5070": centroids.y.astype("float64"),
            "shape_state_name": shapes[state_col].astype(str),
            "shape_county_name": shapes[county_col].astype(str),
        }
    )

    merged = county_rows.merge(shape_keys, on=["state_norm", "county_norm"], how="left")
    centroid_cols = ["county_id", "centroid_x_5070", "centroid_y_5070", "state_name", "Name", "shape_state_name", "shape_county_name"]
    return merged[centroid_cols]


CENTROIDS = load_county_centroids()
WHITELIST_COUNTY_IDS = sorted({r.county_id for r in COUNTY_YEARS})
missing_centroids = CENTROIDS[CENTROIDS["county_id"].isin(WHITELIST_COUNTY_IDS) & CENTROIDS["centroid_x_5070"].isna()]
if len(missing_centroids):
    missing_centroids.to_csv(OUTPUT_UNMATCHED_CENTROIDS_CSV, index=False)
    raise ValueError(f"Missing centroids for {len(missing_centroids)} whitelisted counties. See {OUTPUT_UNMATCHED_CENTROIDS_CSV}")

CENTROID_LOOKUP = CENTROIDS.dropna(subset=["centroid_x_5070", "centroid_y_5070"]).set_index("county_id")[["centroid_x_5070", "centroid_y_5070"]]
CENTROIDS[CENTROIDS["county_id"].isin(WHITELIST_COUNTY_IDS)]

,county_id,centroid_x_5070,centroid_y_5070,state_name,Name,shape_state_name,shape_county_name
100,4_19,471276.277985,1.372895e+06,Arkansas,Cross,Arkansas,Cross
102,4_21,435937.584963,1.207169e+06,Arkansas,Desha,Arkansas,Desha
115,4_34,429618.486314,1.404639e+06,Arkansas,Jackson,Arkansas,Jackson
116,4_35,371638.939655,1.253031e+06,Arkansas,Jefferson,Arkansas,Jefferson
119,4_38,436446.300315,1.454697e+06,Arkansas,Lawrence,Arkansas,Lawrence
...,...,...,...,...,...,...,...
3088,50_69,625952.351240,2.289582e+06,Wisconsin,Washington,Wisconsin,Washington
3089,50_70,623153.646156,2.250211e+06,Wisconsin,Waukesha,Wisconsin,Waukesha
3090,50_71,557830.565038,2.407399e+06,Wisconsin,Waupaca,Wisconsin,Waupaca
3092,50_73,586631.476095,2.364725e+06,Wisconsin,Winnebago,Wisconsin,Winnebago


## Discover And Verify Feature Schema

This reads only file headers. It verifies that the satellite and soil columns are consistent across the whitelisted county-years before the expensive build starts.

`B*_count`, `NDVI_count`, and `GCVI_count` columns are excluded from `sat_*`; one representative count becomes `sat_count_pixels`.

In [8]:
SAT_METADATA_COLUMNS = {"image_id", "longitude", "latitude", "date"}


def header_columns(path: Path) -> list[str]:
    return pd.read_csv(path, nrows=0).columns.tolist()


def source_sat_columns(fused_columns: list[str]) -> list[str]:
    return [
        c for c in fused_columns
        if c not in SAT_METADATA_COLUMNS and not c.endswith("_count")
    ]


def source_count_columns(fused_columns: list[str]) -> list[str]:
    return [c for c in fused_columns if c.endswith("_count")]


def source_soil_columns(soil_columns: list[str]) -> list[str]:
    return [c for c in soil_columns if c.endswith("_mean_1km")]


def discover_feature_schema(records: list[CountyYear]) -> tuple[list[str], list[str], pd.DataFrame]:
    schema_rows = []
    sat_sets = []
    soil_sets = []

    for r in records:
        fused_exists = r.fused_path.exists()
        soil_exists = r.soil_path.exists()
        fused_cols = header_columns(r.fused_path) if fused_exists else []
        soil_cols = header_columns(r.soil_path) if soil_exists else []
        sat_cols = source_sat_columns(fused_cols)
        count_cols = source_count_columns(fused_cols)
        soil_feature_cols = source_soil_columns(soil_cols)

        sat_sets.append(set(sat_cols))
        soil_sets.append(set(soil_feature_cols))
        schema_rows.append(
            {
                "year": r.year,
                "county_id": r.county_id,
                "fused_exists": fused_exists,
                "soil_exists": soil_exists,
                "n_sat_features": len(sat_cols),
                "n_count_columns": len(count_cols),
                "has_preferred_sat_count_source": PREFERRED_SAT_COUNT_SOURCE in count_cols,
                "n_soil_features": len(soil_feature_cols),
            }
        )

    schema_report = pd.DataFrame(schema_rows)
    if not schema_report["fused_exists"].all():
        raise FileNotFoundError("Some whitelisted county-years are missing fused satellite files.")
    if not schema_report["soil_exists"].all():
        raise FileNotFoundError("Some whitelisted county-years are missing static soil-cell files.")

    common_sat = sorted(set.intersection(*sat_sets)) if sat_sets else []
    union_sat = sorted(set.union(*sat_sets)) if sat_sets else []
    common_soil = sorted(set.intersection(*soil_sets)) if soil_sets else []
    union_soil = sorted(set.union(*soil_sets)) if soil_sets else []

    if common_sat != union_sat:
        raise ValueError("Satellite feature columns are not identical across files. Inspect schema_report before relaxing STRICT_REQUIRE_COMMON_SAT_COLUMNS.")
    if common_soil != union_soil:
        raise ValueError("Soil feature columns are not identical across files. Inspect schema_report before relaxing STRICT_REQUIRE_COMMON_SOIL_COLUMNS.")

    sat_source_cols = common_sat if STRICT_REQUIRE_COMMON_SAT_COLUMNS else union_sat
    soil_source_cols = common_soil if STRICT_REQUIRE_COMMON_SOIL_COLUMNS else union_soil
    return sat_source_cols, soil_source_cols, schema_report


SAT_SOURCE_COLS, SOIL_SOURCE_COLS, SCHEMA_REPORT = discover_feature_schema(COUNTY_YEARS)
SAT_OUTPUT_COLS = [f"sat_{c}" for c in SAT_SOURCE_COLS]
SOIL_OUTPUT_COLS = [f"soil_{c}" for c in SOIL_SOURCE_COLS]

print(f"Satellite features: {len(SAT_SOURCE_COLS)}")
print(f"Soil features: {len(SOIL_SOURCE_COLS)}")


def load_patched_soil_points() -> pd.DataFrame:
    """Load the de-duplicated point-level soil patch table, indexed by rounded lon/lat."""
    if not USE_PATCHED_SOIL:
        return pd.DataFrame(columns=SOIL_SOURCE_COLS)
    if not PATCHED_SOIL_CSV.exists():
        raise FileNotFoundError(f"Missing patch file: {PATCHED_SOIL_CSV}")

    patch = pd.read_csv(PATCHED_SOIL_CSV)
    required = ["longitude_debug", "latitude_debug", *SOIL_SOURCE_COLS]
    missing = [c for c in required if c not in patch.columns]
    if missing:
        raise ValueError(f"{PATCHED_SOIL_CSV.name} is missing columns: {missing[:10]}")

    patch = patch[required].copy()
    patch["longitude_debug"] = patch["longitude_debug"].round(JOIN_COORD_DECIMALS)
    patch["latitude_debug"] = patch["latitude_debug"].round(JOIN_COORD_DECIMALS)

    duplicate_points = int(patch.duplicated(["longitude_debug", "latitude_debug"]).sum())
    if duplicate_points:
        print(f"Dropping {duplicate_points} duplicate patched-soil points after rounded lon/lat join.")
        patch = patch.drop_duplicates(["longitude_debug", "latitude_debug"], keep="first")

    return patch.set_index(["longitude_debug", "latitude_debug"])[SOIL_SOURCE_COLS]


PATCHED_SOIL_POINTS = load_patched_soil_points()
print(f"Patched soil points loaded: {len(PATCHED_SOIL_POINTS):,}")
SCHEMA_REPORT.describe(include="all")

Satellite features: 104
Soil features: 18
Patched soil points loaded: 15,535


,year,county_id,fused_exists,soil_exists,n_sat_features,n_count_columns,has_preferred_sat_count_source,n_soil_features
count,1872.000000,1872,1872,1872,1872.0,1872.0,1872,1872.0
unique,NaN,634,1,1,NaN,NaN,1,NaN
top,NaN,35_9,True,True,NaN,NaN,True,NaN
freq,NaN,8,1872,1872,NaN,NaN,1872,NaN
mean,2020.022436,NaN,NaN,NaN,104.0,8.0,NaN,18.0
std,3.121027,NaN,NaN,NaN,0.0,0.0,NaN,0.0
min,2016.000000,NaN,NaN,NaN,104.0,8.0,NaN,18.0
25%,2017.000000,NaN,NaN,NaN,104.0,8.0,NaN,18.0
50%,2020.000000,NaN,NaN,NaN,104.0,8.0,NaN,18.0
75%,2023.000000,NaN,NaN,NaN,104.0,8.0,NaN,18.0


## Build One County-Year Table

This is the main transformation for one county-year:

1. Load static soil cells as the canonical cells.
2. Add rounded debug lon/lat and relative distance to county polygon centroid.
3. Cross every cell with the 14 windows.
4. Load raw fused satellite rows and assign each observation to a window.
5. Average duplicate observations for the same cell-window.
6. Left-join satellite features onto the full cell-window grid.
7. Fill missing satellite windows with zero satellite values and `observed_satellite = 0`.

In [9]:
def assign_windows_to_satellite(fused: pd.DataFrame) -> pd.DataFrame:
    out = fused.copy()
    parsed_dates = out["date"].map(parse_day_month)
    out["window_idx"] = -1

    for row in WINDOWS.itertuples(index=False):
        
        mask = (parsed_dates >= row[3]) & (parsed_dates <= row[4])
        out.loc[mask, "window_idx"] = int(row[0])

    return out[out["window_idx"] >= 0].copy()


def aggregate_satellite_cell_windows(fused: pd.DataFrame) -> pd.DataFrame:
    """Average duplicate satellite observations to one row per cell-window."""
    if fused.empty:
        columns = ["window_idx", "longitude_debug", "latitude_debug", "sat_count_pixels", *SAT_OUTPUT_COLS]
        print("Empty DAtaFRAME Given BAD!!!!")
        return pd.DataFrame(columns=columns)

    fused = add_join_coordinates(fused)
    fused = assign_windows_to_satellite(fused)

    count_source = PREFERRED_SAT_COUNT_SOURCE
    if count_source not in fused.columns:
        print("Missing Column BAD!!!!")


    group_cols = ["window_idx", "longitude_debug", "latitude_debug"]
    agg_source_cols = [count_source, *SAT_SOURCE_COLS]
    available = [c for c in agg_source_cols if c in fused.columns]
    missing = [c for c in agg_source_cols if c not in fused.columns]
    if missing:
        raise ValueError(f"Missing expected satellite columns: {missing[:10]}")

    agg = fused.groupby(group_cols, as_index=False)[available].mean()
    agg = agg.rename(columns={count_source: "sat_count_pixels", **{c: f"sat_{c}" for c in SAT_SOURCE_COLS if c in agg.columns}})

    # Ensure stable final columns even if strict mode was relaxed.
    if "sat_count_pixels" not in agg.columns:
        agg["sat_count_pixels"] = 0.0
    for col in SAT_OUTPUT_COLS:
        if col not in agg.columns:
            agg[col] = 0.0
            raise ValueError(f"Missing expected satellite column: {col}")

    return agg[group_cols + ["sat_count_pixels"] + SAT_OUTPUT_COLS]


def patch_missing_static_soil(soil: pd.DataFrame, manifest: dict) -> pd.DataFrame:
    """Fill missing static soil values from soil_points_filled.csv by rounded lon/lat."""
    if not USE_PATCHED_SOIL:
        return soil

    out = soil.copy()
    before_any = out[SOIL_SOURCE_COLS].isna().any(axis=1)
    before_all = out[SOIL_SOURCE_COLS].isna().all(axis=1)
    patch_values = out[["longitude_debug", "latitude_debug"]].merge(
        PATCHED_SOIL_POINTS.reset_index(),
        on=["longitude_debug", "latitude_debug"],
        how="left",
        validate="many_to_one",
    )
    patch_available = patch_values[SOIL_SOURCE_COLS].notna().any(axis=1)

    for col in SOIL_SOURCE_COLS:
        out[col] = out[col].fillna(patch_values[col])

    after_any = out[SOIL_SOURCE_COLS].isna().any(axis=1)
    after_all = out[SOIL_SOURCE_COLS].isna().all(axis=1)
    positive_incounty = out["static_wheat_pixels_incounty"].fillna(0) >= POSITIVE_INCOUNTY_WHEAT_THRESHOLD

    manifest["static_missing_any_soil_cells_before_patch"] = int(before_any.sum())
    manifest["static_missing_all_soil_cells_before_patch"] = int(before_all.sum())
    manifest["static_missing_soil_patch_rows_available"] = int((before_any & patch_available).sum())
    manifest["static_missing_any_soil_cells_after_patch"] = int(after_any.sum())
    manifest["static_missing_all_soil_cells_after_patch"] = int(after_all.sum())
    manifest["static_positive_incounty_missing_any_soil_cells_after_patch"] = int((positive_incounty & after_any).sum())
    manifest["static_positive_incounty_missing_all_soil_cells_after_patch"] = int((positive_incounty & after_all).sum())
    return out


def make_satellite_only_cells(sat_agg: pd.DataFrame, cells: pd.DataFrame, manifest: dict) -> pd.DataFrame:
    """Create canonical cells for rare satellite coordinates absent from the static soil table."""
    columns = [
        "longitude_debug",
        "latitude_debug",
        "static_wheat_pixels",
        "static_wheat_pixels_incounty",
        "incounty_ratio",
        *SOIL_SOURCE_COLS,
        "x_5070",
        "y_5070",
    ]
    manifest["patched_sat_only_cells_added"] = 0
    manifest["patched_sat_only_cells_without_soil_patch"] = 0
    if not INCLUDE_PATCHED_SAT_ONLY_CELLS or sat_agg.empty:
        return pd.DataFrame(columns=columns)

    sat_cells = sat_agg.groupby(["longitude_debug", "latitude_debug"], as_index=False)["sat_count_pixels"].max()
    existing = cells[["longitude_debug", "latitude_debug"]].drop_duplicates().assign(_in_static=1)
    sat_only = sat_cells.merge(existing, on=["longitude_debug", "latitude_debug"], how="left", validate="one_to_one")
    sat_only = sat_only[sat_only["_in_static"].isna()].drop(columns="_in_static")
    if sat_only.empty:
        print("No missing soil")
        return pd.DataFrame(columns=columns)

    if PATCHED_SOIL_POINTS.empty:
        raise ValueError("Satellite-only cells exist, but PATCHED_SOIL_POINTS is empty.")

    sat_only = sat_only.merge(
        PATCHED_SOIL_POINTS.reset_index(),
        on=["longitude_debug", "latitude_debug"],
        how="left",
        validate="many_to_one",
    )
    missing_patch = sat_only[SOIL_SOURCE_COLS].isna().any(axis=1)
    manifest["patched_sat_only_cells_without_soil_patch"] = int(missing_patch.sum())
    if missing_patch.any():
        examples = sat_only.loc[missing_patch, ["longitude_debug", "latitude_debug"]].head().to_dict(orient="records")
        sat_only = sat_only.loc[~missing_patch].copy()  # 20d: drop unpatchable edge cells instead of raising

    x_5070, y_5070 = LONLAT_TO_5070.transform(
        sat_only["longitude_debug"].to_numpy(),
        sat_only["latitude_debug"].to_numpy(),
    )
    support = sat_only["sat_count_pixels"].fillna(0).round().clip(lower=0, upper=np.iinfo(np.int16).max).astype("int16")

    out = sat_only[["longitude_debug", "latitude_debug", *SOIL_SOURCE_COLS]].copy()
    out["static_wheat_pixels"] = support
    out["static_wheat_pixels_incounty"] = support
    out["incounty_ratio"] = 1.0
    out["x_5070"] = x_5070
    out["y_5070"] = y_5070
    manifest["patched_sat_only_cells_added"] = int(len(out))
    return out[columns]


def downcast_output_table(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["year"] = out["year"].astype("int16")
    out["window_idx"] = out["window_idx"].astype("int8")
    out["observed_satellite"] = out["observed_satellite"].astype("int8")
    out["static_wheat_pixels"] = out["static_wheat_pixels"].astype("int16")
    out["static_wheat_pixels_incounty"] = out["static_wheat_pixels_incounty"].astype("int16")
    out["sat_count_pixels"] = out["sat_count_pixels"].astype("int16")

 
    float32_cols = [
        "longitude_debug",
        "latitude_debug",
        "distance_center_x_km",
        "distance_center_y_km",
        "incounty_ratio",
        *SOIL_OUTPUT_COLS,
        *SAT_OUTPUT_COLS,
        "yield",
    ]
    out[float32_cols] = out[float32_cols].astype("float32")
    return out


def build_county_year_table(record: CountyYear) -> tuple[pd.DataFrame | None, dict]:
    target_yield = YIELD_LOOKUP.get((record.year, record.county_id))
    manifest = {
        "year": record.year,
        "county_id": record.county_id,
        "status": "started",
        "fused_path": str(record.fused_path),
        "soil_path": str(record.soil_path),
        "windowed_path": str(record.windowed_path),
    }

    if target_yield is None:
        manifest["status"] = "skipped_missing_yield"
        return None, manifest

    soil = pd.read_csv(record.soil_path)
    manifest["n_cells_raw"] = int(len(soil))
    if soil.empty:
        manifest["status"] = "skipped_empty_soil"
        print("skipped_empty_soil_county")
        return None, manifest

    centroid = CENTROID_LOOKUP.loc[record.county_id]
    soil = add_join_coordinates(soil)

    required_static_cols = [
        "longitude_debug",
        "latitude_debug",
        "static_wheat_pixels",
        "static_wheat_pixels_incounty",
        "incounty_ratio",
    ]
    missing_required_static_cols = [c for c in required_static_cols if c not in soil.columns]
    if missing_required_static_cols:
        raise ValueError(f"{record.year} {record.county_id}: missing required static columns: {missing_required_static_cols[:10]}")

    # Missing soil values are expected and patched below. If an entire soil feature
    # column is absent from this county-year file, create it as NaN so the same
    # patching path can fill it from soil_points_filled.csv.
    missing_soil_feature_cols = [c for c in SOIL_SOURCE_COLS if c not in soil.columns]
    if missing_soil_feature_cols:
        for col in missing_soil_feature_cols:
            soil[col] = np.nan
        manifest["soil_feature_columns_created_for_patch"] = len(missing_soil_feature_cols)
        manifest["soil_feature_columns_created_for_patch_names"] = ";".join(missing_soil_feature_cols)
    else:
        manifest["soil_feature_columns_created_for_patch"] = 0
        manifest["soil_feature_columns_created_for_patch_names"] = ""

    base_cols = [*required_static_cols, *SOIL_SOURCE_COLS]

    soil = patch_missing_static_soil(soil, manifest)

    positive_incounty_mask = soil["static_wheat_pixels_incounty"].fillna(0) >= POSITIVE_INCOUNTY_WHEAT_THRESHOLD
    missing_any_soil_mask = soil[SOIL_SOURCE_COLS].isna().any(axis=1)
    missing_all_soil_mask = soil[SOIL_SOURCE_COLS].isna().all(axis=1)
    manifest["zero_incounty_cells_kept"] = int((~positive_incounty_mask).sum())
    manifest["missing_any_soil_cells_kept"] = int(missing_any_soil_mask.sum())
    manifest["missing_all_soil_cells_kept"] = int(missing_all_soil_mask.sum())
    manifest["positive_incounty_missing_any_soil_cells"] = int((positive_incounty_mask & missing_any_soil_mask).sum())
    manifest["positive_incounty_missing_all_soil_cells"] = int((positive_incounty_mask & missing_all_soil_mask).sum())

    if soil.empty:
        manifest["status"] = "skipped_empty_soil_after_read"
        return None, manifest

    fused = pd.read_csv(record.fused_path)
    sat_agg = aggregate_satellite_cell_windows(fused)

    # A duplicate after rounding means two soil cells collapse to the same de-facto cell id.
    duplicate_cells = soil.duplicated(["longitude_debug", "latitude_debug"]).sum()
    if duplicate_cells:
        raise ValueError(f"{record.year} {record.county_id}: {duplicate_cells} duplicate rounded soil-cell coordinates")

    cells = soil[base_cols + ["x_5070", "y_5070"]].copy()
    sat_only_cells = make_satellite_only_cells(sat_agg, cells, manifest)
    if len(sat_only_cells):
        cells = pd.concat([cells, sat_only_cells], ignore_index=True)

    duplicate_cells_after_patch = cells.duplicated(["longitude_debug", "latitude_debug"]).sum()
    if duplicate_cells_after_patch:
        raise ValueError(f"{record.year} {record.county_id}: {duplicate_cells_after_patch} duplicate cell coordinates after adding patched satellite-only cells")

    cells["distance_center_x_km"] = (cells["x_5070"] - float(centroid["centroid_x_5070"])) / 1000.0
    cells["distance_center_y_km"] = (cells["y_5070"] - float(centroid["centroid_y_5070"])) / 1000.0
    cells = cells.drop(columns=["x_5070", "y_5070"])
    cells = cells.rename(columns={c: f"soil_{c}" for c in SOIL_SOURCE_COLS})

    # Cross join every canonical cell with every fixed window.
    cells["_cross_key"] = 1
    windows = WINDOWS[["window_idx"]].copy()
    windows["_cross_key"] = 1
    full_grid = cells.merge(windows, on="_cross_key", how="inner").drop(columns="_cross_key")

    out = full_grid.merge(
        sat_agg,
        on=["window_idx", "longitude_debug", "latitude_debug"],
        how="left",
        validate="one_to_one",
    )

    out["observed_satellite"] = out["sat_count_pixels"].notna().astype("int8")
    out["sat_count_pixels"] = out["sat_count_pixels"].fillna(0.0)
    out[SAT_OUTPUT_COLS] = out[SAT_OUTPUT_COLS].fillna(0.0)

    out["year"] = record.year
    out["county_id"] = record.county_id
    out["yield"] = target_yield

    ordered_cols = [
        "year",
        "county_id",
        "window_idx",
        "observed_satellite",
        "longitude_debug",
        "latitude_debug",
        "distance_center_x_km",
        "distance_center_y_km",
        "static_wheat_pixels",
        "static_wheat_pixels_incounty",
        "incounty_ratio",
        "sat_count_pixels",
        *SOIL_OUTPUT_COLS,
        *SAT_OUTPUT_COLS,
        "yield",
    ]
    out = out[ordered_cols]

    out = downcast_output_table(out)
    out = out.sort_values(
        [
            "year",
            "county_id",
            "window_idx",
            "distance_center_x_km",
            "distance_center_y_km",
        ],
        kind="mergesort",
    ).reset_index(drop=True)

    manifest.update(
        {
            "status": "ok",
            "n_static_cells": int(len(soil)),
            "n_cells": int(len(cells)),
            "n_rows": int(len(out)),
            "observed_fraction": float(out["observed_satellite"].mean()),
            "target_yield": float(target_yield),
        }
    )
    return out, manifest

## Dtypes And Sanity Checks

Feature columns are downcast before writing. The sanity checks catch the important invariants before a chunk is appended to the CSV.

In [10]:
def validate_county_year_table(df: pd.DataFrame, record: CountyYear) -> None:
    key = ["year", "county_id", "window_idx", "longitude_debug", "latitude_debug"]
    if df.duplicated(key).any():
        raise ValueError(f"{record.year} {record.county_id}: duplicate cell-window rows")

    rows_per_cell = df.groupby(["year", "county_id", "longitude_debug", "latitude_debug"], observed=True).size()
    bad = rows_per_cell[rows_per_cell != EXPECTED_N_WINDOWS]
    if len(bad):
        raise ValueError(f"{record.year} {record.county_id}: some cells do not have {EXPECTED_N_WINDOWS} rows")

    missing = df["observed_satellite"] == 0
    if not (df.loc[missing, "sat_count_pixels"] == 0).all():
        raise ValueError(f"{record.year} {record.county_id}: missing satellite rows with nonzero sat_count_pixels")
    if SAT_OUTPUT_COLS and not (df.loc[missing, SAT_OUTPUT_COLS] == 0).all(axis=None):
        raise ValueError(f"{record.year} {record.county_id}: missing satellite rows with nonzero sat_* features")

    required_notna = [
        "distance_center_x_km",
        "distance_center_y_km",
        "static_wheat_pixels",
        "static_wheat_pixels_incounty",
        "incounty_ratio",
        "yield",
    ]
    if df[required_notna].isna().any().any():
        raise ValueError(f"{record.year} {record.county_id}: missing required static/distance/yield values")
    # Distances should be stable for the de-facto cell id across all windows.
    distance_variation = df.groupby(["longitude_debug", "latitude_debug"], observed=True)[
        ["distance_center_x_km", "distance_center_y_km"]
    ].nunique()
    if (distance_variation > 1).any().any():
        raise ValueError(f"{record.year} {record.county_id}: distance columns vary across windows for a cell")

## Build CSV And Schema

This is the cell that actually builds `data_sat_soil_20d.csv`. It appends one county-year at a time and writes a manifest plus a schema JSON.

If you want to do a tiny dry run first, uncomment the `records = COUNTY_YEARS[:3]` line.

In [11]:
def write_schema_json(manifest: pd.DataFrame) -> None:
    schema = {
        "dataset": "data_sat_soil_20d",
        "row_definition": "county-year x 10d window x wheat/soil cell",
        "years": YEARS,
        "window_policy": {
            "start": WINDOW_START_DM,
            "end": WINDOW_END_DM,
            "frequency_days": WINDOW_FREQUENCY_DAYS,
            "n_windows": EXPECTED_N_WINDOWS,
            "indexing": "zero_based",
            "windows": WINDOWS[["window_idx", "date_start", "date_end"]].to_dict(orient="records"),
        },
        "county_year_whitelist": "Data_Windowed_{year}_20d",
        "center_definition": "true county polygon centroid from Data_Ashape/gadm36_USA_2.shp, projected to EPSG:5070",
        "coordinate_join_decimals": JOIN_COORD_DECIMALS,
        "missing_satellite_policy": {
            "observed_satellite": 0,
            "sat_count_pixels": 0,
            "sat_features": 0,
            "soil_static_distance_yield": "real values",
        },
        "edge_cell_policy": "keep cells even when static_wheat_pixels_incounty == 0; they are edge/context cells and can be filtered later",
        "patched_soil_source": str(PATCHED_SOIL_CSV) if USE_PATCHED_SOIL else None,
        "missing_soil_policy": "fill missing static soil values from soil_points_filled.csv by rounded lon/lat when available; remaining missing values are counted in the manifest and sanity summary",
        "satellite_only_cell_policy": "if a satellite coordinate is absent from the static soil table, add it when soil_points_filled.csv provides soil; use max sat_count_pixels as static_wheat_pixels and static_wheat_pixels_incounty, with incounty_ratio=1.0",
        "chunk_resume_policy": {
            "reuse_existing_chunks": REUSE_EXISTING_CHUNKS,
            "overwrite_final_outputs_from_chunks": OVERWRITE_FINAL_OUTPUTS_FROM_CHUNKS,
            "final_csv_written_via_temp_file": True,
        },
        "debug_columns_not_for_training": ["longitude_debug", "latitude_debug"],
        "id_columns": ["year", "county_id", "window_idx"],
        "static_columns": ["static_wheat_pixels", "static_wheat_pixels_incounty", "incounty_ratio"],
        "distance_columns": ["distance_center_x_km", "distance_center_y_km"],
        "soil_columns": SOIL_OUTPUT_COLS,
        "satellite_columns": SAT_OUTPUT_COLS,
        "sat_count_pixels_source_preference": PREFERRED_SAT_COUNT_SOURCE,
        "target_column": "yield",
        "manifest_summary": manifest["status"].value_counts(dropna=False).to_dict() if len(manifest) else {},
    }
    OUTPUT_SCHEMA_JSON.write_text(json.dumps(schema, indent=2), encoding="utf-8")


REUSE_EXISTING_CHUNKS = True
OVERWRITE_FINAL_OUTPUTS_FROM_CHUNKS = True


def chunk_path_for(index: int, record: CountyYear) -> Path:
    return OUTPUT_CHUNK_DIR / f"{index:06d}_{record.year}_{record.county_id}.csv"


def build_one_chunk(index_and_record: tuple[int, CountyYear]) -> dict:
    """Build one county-year chunk, or reuse an existing non-empty chunk CSV."""
    index, record = index_and_record
    chunk_path = chunk_path_for(index, record)

    if REUSE_EXISTING_CHUNKS and chunk_path.exists() and chunk_path.stat().st_size > 0:
        return {
            "year": record.year,
            "county_id": record.county_id,
            "status": "ok_reused_chunk",
            "fused_path": str(record.fused_path),
            "soil_path": str(record.soil_path),
            "windowed_path": str(record.windowed_path),
            "chunk_path": str(chunk_path),
            "build_index": index,
        }

    table, manifest = build_county_year_table(record)
    manifest["chunk_path"] = str(chunk_path)
    manifest["build_index"] = index

    if table is not None:
        validate_county_year_table(table, record)
        table.to_csv(chunk_path, index=False)

    return manifest


def add_chunk_summary_to_manifest(manifest: dict, chunk: pd.DataFrame) -> dict:
    """Recover important manifest fields for reused chunks while they are being merged."""
    out = dict(manifest)
    cell_cols = ["longitude_debug", "latitude_debug"]
    positive_incounty = chunk["static_wheat_pixels_incounty"].fillna(0) >= POSITIVE_INCOUNTY_WHEAT_THRESHOLD
    soil_missing_any = chunk[SOIL_OUTPUT_COLS].isna().any(axis=1) if SOIL_OUTPUT_COLS else pd.Series(False, index=chunk.index)
    soil_missing_all = chunk[SOIL_OUTPUT_COLS].isna().all(axis=1) if SOIL_OUTPUT_COLS else pd.Series(False, index=chunk.index)

    out["n_rows"] = int(len(chunk))
    out["n_cells"] = int(chunk[cell_cols].drop_duplicates().shape[0])
    out["observed_fraction"] = float(chunk["observed_satellite"].mean()) if len(chunk) else 0.0
    out["target_yield"] = float(chunk["yield"].iloc[0]) if len(chunk) else np.nan
    out["zero_incounty_cells_kept"] = int(chunk.loc[~positive_incounty, cell_cols].drop_duplicates().shape[0])
    out["missing_any_soil_cells_kept"] = int(chunk.loc[soil_missing_any, cell_cols].drop_duplicates().shape[0])
    out["missing_all_soil_cells_kept"] = int(chunk.loc[soil_missing_all, cell_cols].drop_duplicates().shape[0])
    out["positive_incounty_missing_any_soil_cells"] = int(chunk.loc[positive_incounty & soil_missing_any, cell_cols].drop_duplicates().shape[0])
    out["positive_incounty_missing_all_soil_cells"] = int(chunk.loc[positive_incounty & soil_missing_all, cell_cols].drop_duplicates().shape[0])
    return out


def build_csv(records: list[CountyYear]) -> pd.DataFrame:
    output_tmp = OUTPUT_CSV.with_name(OUTPUT_CSV.name + ".tmp")

    if (OUTPUT_CSV.exists() or OUTPUT_MANIFEST_CSV.exists() or OUTPUT_SCHEMA_JSON.exists()) and not OVERWRITE_FINAL_OUTPUTS_FROM_CHUNKS:
        raise FileExistsError("Final outputs already exist. Set OVERWRITE_FINAL_OUTPUTS_FROM_CHUNKS=True to rebuild them from chunks.")
    if OUTPUT_CHUNK_DIR.exists() and any(OUTPUT_CHUNK_DIR.iterdir()) and not REUSE_EXISTING_CHUNKS:
        raise FileExistsError(f"Refusing to reuse non-empty chunk directory: {OUTPUT_CHUNK_DIR}")

    OUTPUT_CHUNK_DIR.mkdir(parents=True, exist_ok=True)
    if output_tmp.exists():
        output_tmp.unlink()

    indexed_records = list(enumerate(records))
    manifest_rows = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(build_one_chunk, item) for item in indexed_records]
        for i, future in enumerate(as_completed(futures), start=1):
            manifest_rows.append(future.result())
            if i % PROGRESS_EVERY == 0 or i == len(futures):
                reused = sum(m.get("status") == "ok_reused_chunk" for m in manifest_rows)
                built = sum(m.get("status") == "ok" for m in manifest_rows)
                print(f"Prepared {i:,}/{len(futures):,} chunks ({built:,} built, {reused:,} reused)")

    manifest_df = pd.DataFrame(manifest_rows).sort_values("build_index").reset_index(drop=True)

    wrote_header = False
    final_manifest_rows = []
    ok_statuses = {"ok", "ok_reused_chunk"}
    ok_rows = manifest_df[manifest_df["status"].isin(ok_statuses)].sort_values("build_index")
    for i, row in enumerate(ok_rows.itertuples(index=False), start=1):
        chunk_path = Path(getattr(row, "chunk_path"))
        chunk = pd.read_csv(chunk_path)
        chunk.to_csv(output_tmp, mode="a", index=False, header=not wrote_header)
        wrote_header = True
        final_manifest_rows.append(add_chunk_summary_to_manifest(row._asdict(), chunk))
        if i % PROGRESS_EVERY == 0 or i == len(ok_rows):
            print(f"Merged {i:,}/{len(ok_rows):,} chunks into {output_tmp.name}")

    skipped_manifest_rows = [row._asdict() for row in manifest_df[~manifest_df["status"].isin(ok_statuses)].itertuples(index=False)]
    final_manifest = pd.DataFrame(final_manifest_rows + skipped_manifest_rows).sort_values("build_index").reset_index(drop=True)
    final_manifest.to_csv(OUTPUT_MANIFEST_CSV, index=False)
    write_schema_json(final_manifest)
    output_tmp.replace(OUTPUT_CSV)
    return final_manifest

# For a dry run, uncomment one of these lines and comment the full build line:
# records_to_build = COUNTY_YEARS[:3]
# records_to_build = COUNTY_YEARS[:100]
records_to_build = COUNTY_YEARS
manifest = build_csv(records_to_build)
manifest["status"].value_counts(dropna=False)


Prepared 50/1,872 chunks (0 built, 50 reused)
Prepared 100/1,872 chunks (0 built, 100 reused)
Prepared 150/1,872 chunks (0 built, 150 reused)
Prepared 200/1,872 chunks (0 built, 200 reused)
Prepared 250/1,872 chunks (0 built, 250 reused)
Prepared 300/1,872 chunks (0 built, 300 reused)
Prepared 350/1,872 chunks (0 built, 350 reused)
Prepared 400/1,872 chunks (0 built, 400 reused)
Prepared 450/1,872 chunks (0 built, 450 reused)
Prepared 500/1,872 chunks (0 built, 500 reused)
Prepared 550/1,872 chunks (0 built, 550 reused)
Prepared 600/1,872 chunks (0 built, 600 reused)
Prepared 650/1,872 chunks (0 built, 650 reused)
Prepared 700/1,872 chunks (0 built, 700 reused)
Prepared 750/1,872 chunks (0 built, 750 reused)
Prepared 800/1,872 chunks (0 built, 800 reused)
Prepared 850/1,872 chunks (0 built, 850 reused)
Prepared 900/1,872 chunks (0 built, 900 reused)
Prepared 950/1,872 chunks (0 built, 950 reused)
Prepared 1,000/1,872 chunks (0 built, 1,000 reused)
Prepared 1,050/1,872 chunks (0 built, 

status
ok_reused_chunk    1872
Name: count, dtype: int64

In [16]:
manifest[
    [
        "year",
        "county_id",
        "status",
        "n_cells",
        "zero_incounty_cells_kept",
        "missing_any_soil_cells_kept",
        "positive_incounty_missing_any_soil_cells",
        "positive_incounty_missing_all_soil_cells",
        "observed_fraction",
    ]
].sort_values(
    "positive_incounty_missing_any_soil_cells",
    ascending=False
).head(20)

,year,county_id,status,n_cells,zero_incounty_cells_kept,missing_any_soil_cells_kept,positive_incounty_missing_any_soil_cells,positive_incounty_missing_all_soil_cells,observed_fraction
0,2016,5_39,ok_reused_chunk,1132,26,4,0,0,0.846605
1,2016,6_1,ok_reused_chunk,2386,29,3,0,0,0.786702
2,2016,6_22,ok_reused_chunk,960,16,0,0,0,0.752158
3,2016,6_32,ok_reused_chunk,2039,24,3,0,0,0.667764
4,2016,6_33,ok_reused_chunk,3890,23,2,0,0,0.752387
5,2016,6_38,ok_reused_chunk,2240,16,3,0,0,0.710236
6,2016,6_51,ok_reused_chunk,2262,19,2,0,0,0.606606
7,2016,6_62,ok_reused_chunk,4274,27,6,0,0,0.660154
8,2016,14_1,ok_reused_chunk,266,11,1,0,0,0.652793
9,2016,14_7,ok_reused_chunk,25,2,0,0,0,0.525714


## Read-Only Sanity Pass On The Finished CSV

This cell reads the finished CSV and checks the global invariants. It is intentionally separate from the build cell so you can run it after the CSV exists. Missing soil is not treated as an automatic failure here; it is reported, especially for cells with positive in-county wheat support.

In [14]:
df = pd.read_csv(OUTPUT_CSV)

key = ["year", "county_id", "window_idx", "longitude_debug", "latitude_debug"]
duplicate_count = int(df.duplicated(key).sum())

rows_per_cell = df.groupby(["year", "county_id", "longitude_debug", "latitude_debug"], observed=True).size()
bad_cell_row_counts = rows_per_cell[rows_per_cell != EXPECTED_N_WINDOWS]

missing = df["observed_satellite"] == 0
missing_sat_count_pixels_bad = int((df.loc[missing, "sat_count_pixels"] != 0).sum())
missing_sat_feature_bad = int((df.loc[missing, SAT_OUTPUT_COLS] != 0).any(axis=1).sum()) if SAT_OUTPUT_COLS else 0
positive_incounty = df["static_wheat_pixels_incounty"].fillna(0) >= POSITIVE_INCOUNTY_WHEAT_THRESHOLD
soil_missing_any = df[SOIL_OUTPUT_COLS].isna().any(axis=1) if SOIL_OUTPUT_COLS else pd.Series(False, index=df.index)
soil_missing_all = df[SOIL_OUTPUT_COLS].isna().all(axis=1) if SOIL_OUTPUT_COLS else pd.Series(False, index=df.index)

bag_sizes = rows_per_cell.groupby(level=[0, 1]).size().rename("n_cells").reset_index()
observed_by_bag = df.groupby(["year", "county_id"], observed=True)["observed_satellite"].sum().rename("observed_cell_windows").reset_index()
degenerate_bags = observed_by_bag[observed_by_bag["observed_cell_windows"] == 0]

summary = {
    "rows": len(df),
    "county_years": df[["year", "county_id"]].drop_duplicates().shape[0],
    "duplicate_cell_windows": duplicate_count,
    "cells_with_wrong_window_count": len(bad_cell_row_counts),
    "missing_rows_with_nonzero_sat_count_pixels": missing_sat_count_pixels_bad,
    "missing_rows_with_nonzero_sat_features": missing_sat_feature_bad,
    "degenerate_bags_zero_observed_satellite": len(degenerate_bags),
    "cells_with_any_missing_soil": int(df.loc[soil_missing_any, ["year", "county_id", "longitude_debug", "latitude_debug"]].drop_duplicates().shape[0]),
    "positive_incounty_cells_with_any_missing_soil": int(df.loc[soil_missing_any & positive_incounty, ["year", "county_id", "longitude_debug", "latitude_debug"]].drop_duplicates().shape[0]),
    "positive_incounty_cells_with_all_missing_soil": int(df.loc[soil_missing_all & positive_incounty, ["year", "county_id", "longitude_debug", "latitude_debug"]].drop_duplicates().shape[0]),
    "bag_size_min": int(bag_sizes["n_cells"].min()),
    "bag_size_median": float(bag_sizes["n_cells"].median()),
    "bag_size_max": int(bag_sizes["n_cells"].max()),
}

summary

{'rows': 21139846,
 'county_years': 1872,
 'duplicate_cell_windows': 0,
 'cells_with_wrong_window_count': 0,
 'missing_rows_with_nonzero_sat_count_pixels': 0,
 'missing_rows_with_nonzero_sat_features': 0,
 'degenerate_bags_zero_observed_satellite': 0,
 'cells_with_any_missing_soil': 2858,
 'positive_incounty_cells_with_any_missing_soil': 0,
 'positive_incounty_cells_with_all_missing_soil': 0,
 'bag_size_min': 1,
 'bag_size_median': 411.0,
 'bag_size_max': 4883}

## Build Parquet From CSV

This final cell converts the finished CSV to Parquet. It requires either `pyarrow` or `fastparquet` in the active notebook environment.

The CSV remains the inspectable source artifact; Parquet is for faster loading and smaller storage.

In [15]:
df = pd.read_csv(OUTPUT_CSV)

# Reapply compact dtypes after reading from CSV so Parquet preserves them.
df["year"] = df["year"].astype("int16")
df["window_idx"] = df["window_idx"].astype("int8")
df["observed_satellite"] = df["observed_satellite"].astype("int8")

float32_cols = [
    "longitude_debug",
    "latitude_debug",
    "distance_center_x_km",
    "distance_center_y_km",
    "static_wheat_pixels",
    "static_wheat_pixels_incounty",
    "incounty_ratio",
    "sat_count_pixels",
    *SOIL_OUTPUT_COLS,
    *SAT_OUTPUT_COLS,
    "yield",
]
df[float32_cols] = df[float32_cols].astype("float32")

df.to_parquet(OUTPUT_PARQUET, index=False)
OUTPUT_PARQUET

WindowsPath('<repository>/data_sat_soil_20d.parquet')